# AI Pneumonia Detection from Chest X-Rays using VGG16

This notebook builds a beginner-friendly deep learning project that detects whether a chest X-ray image is **NORMAL** or shows **PNEUMONIA**.

The project uses **transfer learning** with **VGG16**, a convolutional neural network that was already trained on a very large image dataset called ImageNet. Instead of training a CNN from scratch, we reuse VGG16 as a feature extractor and add our own classification layers on top.


## What this project does

The goal of this project is to teach a computer to look at chest X-ray images and classify each image into one of two categories:

1. **NORMAL**: no pneumonia detected in the X-ray image.
2. **PNEUMONIA**: pneumonia patterns are present in the X-ray image.

In simple terms, the model learns visual patterns. During training, it sees many labeled X-rays. Over time, it learns which image features are more common in normal lungs and which features are more common in pneumonia cases. After training, we can give it a new X-ray image and ask it to predict the class.


## Dataset

This notebook is designed for the Kaggle dataset:

**Chest X-Ray Images (Pneumonia)**  
Common Kaggle link: `https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia`

The dataset is usually organized like this:

```text
chest_xray/
  train/
    NORMAL/
    PNEUMONIA/
  val/
    NORMAL/
    PNEUMONIA/
  test/
    NORMAL/
    PNEUMONIA/
```

If you run this on Kaggle, the path is often:

```text
/kaggle/input/chest-xray-pneumonia/chest_xray/
```

If you run this locally, download the dataset and set `DATA_DIR` to the path of the `chest_xray` folder.


In [1]:
# If you are running locally, install these first in your terminal:
# pip install tensorflow matplotlib numpy scikit-learn

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 1: Set the dataset path

Change `DATA_DIR` if your dataset is somewhere else.

The folder should contain `train`, `val`, and `test` folders.


In [2]:
# Kaggle path:
DATA_DIR = "/kaggle/input/chest-xray-pneumonia/chest_xray"

# Local example:
# DATA_DIR = r"C:\Users\YourName\Downloads\chest_xray"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

print("Train path exists:", os.path.exists(TRAIN_DIR))
print("Val path exists:", os.path.exists(VAL_DIR))
print("Test path exists:", os.path.exists(TEST_DIR))


Train path exists: False
Val path exists: False
Test path exists: False


## Step 2: Load the images

VGG16 expects images to be resized to **224 × 224** pixels. We also use a batch size of 32, meaning the model processes 32 images at a time during training.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Class names:", class_names)


## Step 3: Speed up the data pipeline

`cache()` keeps images ready after they are loaded once.  
`prefetch()` prepares the next batch while the model is training on the current batch.

This makes training faster.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## Step 4: Visualize sample X-rays


In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_index = int(labels[i].numpy()[0])
        plt.title(class_names[label_index])
        plt.axis("off")


## Step 5: Data augmentation

Data augmentation creates slightly modified versions of training images. For example, the model may see an image slightly rotated or zoomed.

This helps reduce overfitting because the model learns more general visual patterns instead of memorizing the exact training images.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")


## Step 6: Build the VGG16 transfer learning model

We use VGG16 without its original final classification layer.

The original VGG16 model was trained to classify 1000 ImageNet categories, such as cats, dogs, cars, and airplanes. We do not need those categories. We only need two categories: **NORMAL** and **PNEUMONIA**.

So we:
1. Load VGG16 with ImageNet weights.
2. Remove the original classifier using `include_top=False`.
3. Freeze the VGG16 base so its learned filters do not change at first.
4. Add our own classifier for pneumonia detection.


In [ ]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)
x = preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.summary()


## Step 7: Compile the model

Because this is a binary classification problem, we use:

- **Binary cross-entropy** as the loss function.
- **Adam** as the optimizer.
- **Accuracy**, **precision**, and **recall** as metrics.

Recall is especially important for medical screening-style projects because we care about catching positive pneumonia cases.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)


## Step 8: Train the model

The callbacks help training:

- `EarlyStopping`: stops training if validation loss stops improving.
- `ModelCheckpoint`: saves the best model.
- `ReduceLROnPlateau`: lowers the learning rate if the model gets stuck.


In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "best_vgg16_pneumonia_model.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


## Step 9: Plot training results

These graphs show how the model improved during training.

- If training accuracy goes up but validation accuracy stays low, the model may be overfitting.
- If both training and validation improve, the model is learning useful patterns.
- If loss decreases, the model is making fewer severe mistakes.


In [ ]:
def plot_history(history):
    metrics = ["loss", "accuracy", "precision", "recall"]

    for metric in metrics:
        plt.figure(figsize=(8, 5))
        plt.plot(history.history[metric], label=f"Training {metric}")
        plt.plot(history.history[f"val_{metric}"], label=f"Validation {metric}")
        plt.xlabel("Epoch")
        plt.ylabel(metric.title())
        plt.title(f"Training vs Validation {metric.title()}")
        plt.legend()
        plt.grid(True)
        plt.show()

plot_history(history)


## Step 10: Evaluate on the test set

The test set contains images the model did not train on. This gives a better estimate of how well the model handles new images.


In [ ]:
test_results = model.evaluate(test_ds, verbose=1)

for name, value in zip(model.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

## Step 11: Confusion matrix and classification report

A confusion matrix shows the types of mistakes the model makes.

For binary pneumonia detection:

- **True Negative**: normal X-ray predicted as normal.
- **False Positive**: normal X-ray predicted as pneumonia.
- **False Negative**: pneumonia X-ray predicted as normal.
- **True Positive**: pneumonia X-ray predicted as pneumonia.

False negatives are especially serious in a medical screening context because the model misses a pneumonia case.


In [ ]:
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred_probs.extend(probs.flatten())
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true).astype(int)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(6, 6))
disp.plot(cmap=None)
plt.title("Confusion Matrix")
plt.show()


## Step 12: Predict one image

Use this function to test the model on a single image file.

The model outputs a probability between 0 and 1.

- Close to 0 means the model thinks the image is more likely **NORMAL**.
- Close to 1 means the model thinks the image is more likely **PNEUMONIA**.


In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

def predict_xray(image_path, model, class_names):
    img = load_img(image_path, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    probability = model.predict(img_array, verbose=0)[0][0]
    predicted_index = int(probability >= 0.5)
    predicted_class = class_names[predicted_index]

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Prediction: {predicted_class}\nPneumonia probability: {probability:.2%}")
    plt.show()

    return predicted_class, probability

# Example:
# image_path = "/kaggle/input/chest-xray-pneumonia/chest_xray/test/PNEUMONIA/person1_virus_6.jpeg"
# predict_xray(image_path, model, class_names)


## Optional Step 13: Fine-tune the model

After the top classifier has trained, you can unfreeze some of the later VGG16 layers and train them slowly.

Fine-tuning can improve performance, but it can also overfit if the learning rate is too high.

Run this only after the first training stage works.


In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze earlier layers and only train later layers
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)


## References

- Kaggle Chest X-Ray Images Pneumonia dataset: `https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia`
- TensorFlow VGG16 documentation: `https://www.tensorflow.org/api_docs/python/tf/keras/applications/VGG16`
- Keras Applications documentation: `https://keras.io/api/applications/`
